In [ ]:
# Finalised models

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import joblib


In [2]:
# Load datasets

nb = pd.read_csv("../Data/netbanking_unauthorized.csv")
upi = pd.read_csv("../Data/upi_unauthorized.csv")
nd = pd.read_csv("../Data/non_delivery.csv")

print("Netbanking:", nb.shape)
print("UPI:", upi.shape)
print("Non-Delivery:", nd.shape)

Netbanking: (1000, 15)
UPI: (1200, 15)
Non-Delivery: (1400, 15)


In [3]:
# Define features and target

# Netbanking
X_nb = nb.drop(columns=[
    "case_id",
    "reason_code",
    "payment_rail",
    "delivery_confirmed",
    "tracking_available",
    "label"
])
y_nb = nb["label"]

# UPI
X_upi = upi.drop(columns=[
    "case_id",
    "reason_code",
    "payment_rail",
    "delivery_confirmed",
    "tracking_available",
    "label"
])
y_upi = upi["label"]

# Non-Delivery
X_nd = nd.drop(columns=[
    "case_id",
    "reason_code",
    "payment_rail",
    "device_ip_match_history",
    "auth_flow_type",
    "label"
])
y_nd = nd["label"]

print("Netbanking features:", X_nb.columns.tolist())
print("UPI features:", X_upi.columns.tolist())
print("Non-Delivery features:", X_nd.columns.tolist())

Netbanking features: ['order_value', 'days_since_transaction', 'device_ip_match_history', 'auth_flow_type', 'customer_account_age_days', 'customer_past_order_count', 'customer_past_dispute_count', 'merchant_comm_log_exists', 'refund_already_issued']
UPI features: ['order_value', 'days_since_transaction', 'device_ip_match_history', 'auth_flow_type', 'customer_account_age_days', 'customer_past_order_count', 'customer_past_dispute_count', 'merchant_comm_log_exists', 'refund_already_issued']
Non-Delivery features: ['order_value', 'days_since_transaction', 'customer_account_age_days', 'customer_past_order_count', 'customer_past_dispute_count', 'delivery_confirmed', 'tracking_available', 'merchant_comm_log_exists', 'refund_already_issued']


In [4]:
# Train-test split

X_train_nb, X_test_nb, y_train_nb, y_test_nb = train_test_split(
    X_nb, y_nb,
    test_size=0.20,
    random_state=42,
    stratify=y_nb
)

X_train_upi, X_test_upi, y_train_upi, y_test_upi = train_test_split(
    X_upi, y_upi,
    test_size=0.20,
    random_state=42,
    stratify=y_upi
)

X_train_nd, X_test_nd, y_train_nd, y_test_nd = train_test_split(
    X_nd, y_nd,
    test_size=0.20,
    random_state=42,
    stratify=y_nd
)

print("Netbanking:", X_train_nb.shape, X_test_nb.shape)
print("UPI:", X_train_upi.shape, X_test_upi.shape)
print("Non-Delivery:", X_train_nd.shape, X_test_nd.shape)

Netbanking: (800, 9) (200, 9)
UPI: (960, 9) (240, 9)
Non-Delivery: (1120, 9) (280, 9)


In [5]:
# Define preprocessing pipelines

# Netbanking and UPI
numeric_features_auth = [
    "order_value",
    "days_since_transaction",
    "device_ip_match_history",
    "customer_account_age_days",
    "customer_past_order_count",
    "customer_past_dispute_count",
    "merchant_comm_log_exists",
    "refund_already_issued"
]

categorical_features_auth = [
    "auth_flow_type"
]

numeric_transformer_auth = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer_auth = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_auth = ColumnTransformer([
    ("num", numeric_transformer_auth, numeric_features_auth),
    ("cat", categorical_transformer_auth, categorical_features_auth)
])


# Non-Delivery
numeric_features_nd = [
    "order_value",
    "days_since_transaction",
    "customer_account_age_days",
    "customer_past_order_count",
    "customer_past_dispute_count",
    "delivery_confirmed",
    "tracking_available",
    "merchant_comm_log_exists",
    "refund_already_issued"
]

preprocessor_nd = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features_nd)
])

In [8]:
# Create separate preprocessing pipelines for UPI and Netbanking

preprocessor_upi = ColumnTransformer([
    ("num", numeric_transformer_auth, numeric_features_auth),
    ("cat", categorical_transformer_auth, categorical_features_auth)
])

preprocessor_nb = ColumnTransformer([
    ("num", numeric_transformer_auth, numeric_features_auth),
    ("cat", categorical_transformer_auth, categorical_features_auth)
])

print("Separate preprocessors created successfully!")

Separate preprocessors created successfully!


In [9]:
# Define and train final models

# 1. UPI Unauthorized → Random Forest
rf_upi = Pipeline([
    ("preprocessor", preprocessor_upi),
    ("model", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        class_weight="balanced",
        min_samples_leaf=2,
        n_jobs=-1
    ))
])

rf_upi.fit(X_train_upi, y_train_upi)


# 2. Netbanking Unauthorized → Random Forest
rf_nb = Pipeline([
    ("preprocessor", preprocessor_nb),
    ("model", RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        class_weight="balanced",
        min_samples_leaf=2,
        n_jobs=-1
    ))
])

rf_nb.fit(X_train_nb, y_train_nb)


# 3. Non-Delivery → Logistic Regression
lr_nd = Pipeline([
    ("preprocessor", preprocessor_nd),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

lr_nd.fit(X_train_nd, y_train_nd)

print("All 3 final models trained successfully!")

All 3 final models trained successfully!


In [10]:
# Evaluate final models

final_models = {
    "UPI Unauthorized - Random Forest": (rf_upi, X_test_upi, y_test_upi),
    "Netbanking Unauthorized - Random Forest": (rf_nb, X_test_nb, y_test_nb),
    "Non-Delivery - Logistic Regression": (lr_nd, X_test_nd, y_test_nd)
}

for name, (model, X_test, y_test) in final_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("Accuracy :", round(accuracy_score(y_test, y_pred), 4))
    print("Precision:", round(precision_score(y_test, y_pred), 4))
    print("Recall   :", round(recall_score(y_test, y_pred), 4))
    print("F1 Score :", round(f1_score(y_test, y_pred), 4))
    print("ROC-AUC  :", round(roc_auc_score(y_test, y_prob), 4))


UPI Unauthorized - Random Forest
Accuracy : 0.9167
Precision: 0.9422
Recall   : 0.9422
F1 Score : 0.9422
ROC-AUC  : 0.9306

Netbanking Unauthorized - Random Forest
Accuracy : 0.86
Precision: 0.8824
Recall   : 0.8824
F1 Score : 0.8824
ROC-AUC  : 0.9109

Non-Delivery - Logistic Regression
Accuracy : 0.8429
Precision: 0.9669
Recall   : 0.8216
F1 Score : 0.8883
ROC-AUC  : 0.9047


In [ ]:
# Model finalisation done and now saving


In [11]:
# Save final trained models

import os

os.makedirs("models", exist_ok=True)

joblib.dump(rf_upi, "models/upi_rf.pkl")
joblib.dump(rf_nb, "models/netbanking_rf.pkl")
joblib.dump(lr_nd, "models/non_delivery_lr.pkl")

print("All 3 models saved successfully!")

All 3 models saved successfully!


In [12]:
# Test loading a saved model

loaded_upi_model = joblib.load("models/upi_rf.pkl")

print("UPI model loaded successfully!")
print(type(loaded_upi_model))

UPI model loaded successfully!
<class 'sklearn.pipeline.Pipeline'>


In [13]:
# Test prediction using the saved UPI model

sample = X_test_upi.iloc[[0]]

prediction = loaded_upi_model.predict(sample)[0]
probability = loaded_upi_model.predict_proba(sample)[0][1]

print("Prediction:", prediction)
print("Fight probability:", round(probability * 100, 2), "%")

Prediction: 0
Fight probability: 14.92 %


In [14]:
# Verify all saved models can be loaded 

loaded_upi_model = joblib.load("models/upi_rf.pkl")
loaded_nb_model = joblib.load("models/netbanking_rf.pkl")
loaded_nd_model = joblib.load("models/non_delivery_lr.pkl")

print("UPI model:", type(loaded_upi_model))
print("Netbanking model:", type(loaded_nb_model))
print("Non-Delivery model:", type(loaded_nd_model))

UPI model: <class 'sklearn.pipeline.Pipeline'>
Netbanking model: <class 'sklearn.pipeline.Pipeline'>
Non-Delivery model: <class 'sklearn.pipeline.Pipeline'>


In [15]:
# Test all 3 saved models

tests = [
    ("UPI", loaded_upi_model, X_test_upi.iloc[[0]]),
    ("Netbanking", loaded_nb_model, X_test_nb.iloc[[0]]),
    ("Non-Delivery", loaded_nd_model, X_test_nd.iloc[[0]])
]

for name, model, sample in tests:
    prediction = model.predict(sample)[0]
    probability = model.predict_proba(sample)[0][1]

    decision = "Fight" if prediction == 1 else "Don't Fight"

    print(f"\n{name}")
    print("-" * 30)
    print("Decision:", decision)
    print("Fight probability:", round(probability * 100, 2), "%")


UPI
------------------------------
Decision: Don't Fight
Fight probability: 14.92 %

Netbanking
------------------------------
Decision: Fight
Fight probability: 95.3 %

Non-Delivery
------------------------------
Decision: Fight
Fight probability: 72.51 %
